# VAYU Climate Digital Twin — Kaggle GPU Training (Full India)

**Accelerator**: GPU T4 x2 (recommended)  
**Target**: R2_tmax >= 0.80, R2_rain 0.30-0.40 — Full India 2010-2025

## IMPORTANT — this region is different from the other 4
Full India has **15,367 nodes** vs ~1,200-2,200 for Western Ghats / Central India /
North-East India / Indo-Gangetic Plain (roughly 7-12x larger graph). Prior local runs
OOM'd at the default architecture on this node count. This notebook uses the
`--kaggle-medium` preset (smaller GNN hidden dim / transformer d_model, batch=1)
which was validated in `research/PS5_IMPLEMENTATION_PLAN.md` to run without OOM on
T4. `--run-baselines` (RandomForest/XGBoost) is intentionally **skipped** here —
at 15,367 nodes x 512 sequences that's ~7.9M rows and can stall/OOM on Kaggle's
CPU; persistence and climatology baselines still run automatically as part of
every training epoch's evaluation, so you still get those comparisons for free.

**If you hit OOM even with kaggle_medium**, drop to `--kaggle-lite` instead (see
commented alternative in the training cell) — smaller still, at some accuracy cost.

## Region priority: balanced (no override)
Unlike the 4 specialist regions, Full India is the composite/aggregation model and
is intentionally left at the global default loss weights (rainfall=1.8, temp_max=1.6,
temp_min=1.2) rather than biased toward any single region's dominant hazard —
upweighting one variable nationally would trade off skill in regions where that
variable is not the priority (e.g. boosting rainfall weight would hurt IGP's heat-
extreme skill). No CLI weight override is passed in the training cell below.

## Required Dataset (Add Input -> Search by name)
**`shyam31415/vayu-full-india-processed`** (or whichever name you use when you
manually create the Full India dataset) — must contain:
- train_sequences.pt, val_sequences.pt (17 features/node)
- normalized_2010-2025.nc, sequence_manifest.json, pipeline_log_2010-2025.json

Only 2010-2011 has real ERA5 wind/humidity for this region; remaining years are
zero-filled. NaN fraction in tmax/tmin is ~6.3% (likely ocean/border cells outside
IMD land coverage) — handled by build-sequences' fillna(0.0).

## Steps
1. Enable GPU: Settings -> Accelerator -> **GPU T4 x2**
2. Add the dataset above via "Add Input" (update the dataset ref name in the cell
   below to match whatever you name it when uploading manually)
3. Run all cells top to bottom

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
import sys, os
from pathlib import Path

REGION = 'full_india'
REPO_DIR = '/kaggle/working/isro'
PROCESSED_DIR = f'{REPO_DIR}/data/processed_{REGION}'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/{REGION}_main'

if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'rm -rf {REPO_DIR}')
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# NOTE: must run after clone/rm -rf above, since these dirs are nested inside
# REPO_DIR and would otherwise be wiped out by rm -rf.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

required = ['train_sequences.pt', 'val_sequences.pt', 'normalized_2010-2025.nc', 'sequence_manifest.json']
root = Path('/kaggle/input')
found = {name: next(iter(root.rglob(name)), None) for name in required}
missing = [k for k, v in found.items() if v is None]
if missing:
    raise RuntimeError('Missing dataset files: ' + ', '.join(missing) + ' — attach the Full India dataset via \'Add Input\'.')
parent_counts = {}
for p in found.values():
    parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
DATASET_DIR = max(parent_counts, key=parent_counts.get)
print('Dataset dir:', DATASET_DIR)

In [ ]:
import shutil
from pathlib import Path as _P

for f in ['train_sequences.pt', 'val_sequences.pt', 'normalized_2010-2025.nc',
          'sequence_manifest.json', 'pipeline_log_2010-2025.json']:
    src = _P(DATASET_DIR) / f
    if src.exists():
        shutil.copy(src, PROCESSED_DIR)
        print(f'copied {f}')
    else:
        print(f'skip (not in bundle): {f}')

os.system(f'ls -lah {PROCESSED_DIR}')

In [ ]:
# ── Smoke check with kaggle_medium preset (must match the training cell's preset) ──
import subprocess, sys, torch
PY = sys.executable

_seq_path = f'{PROCESSED_DIR}/train_sequences.pt'
_seqs = torch.load(_seq_path, map_location='cpu', weights_only=False)
_nf = _seqs[0][0].x.shape[-1]
_nn = _seqs[0][0].x.shape[0]
print(f'Sequence feature count: {_nf} (expected 17) | nodes: {_nn} (expected 15367)')
if _nf != 17:
    raise RuntimeError(f'STALE SEQUENCES: found {_nf} features, expected 17.')
del _seqs
print('✓ Sequences verified')

_r = subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir', PROCESSED_DIR,
    '--checkpoint-dir', f'{REPO_DIR}/checkpoints/{REGION}_smoke',
    '--epochs', '1', '--device', 'auto', '--smoke-only',
    '--kaggle-medium'],
    cwd=REPO_DIR, capture_output=True, text=True)
print(_r.stdout[-3000:] if _r.stdout else '')
if _r.returncode != 0:
    print('\n=== smoke STDERR ===')
    print(_r.stderr[-4000:] if _r.stderr else '(empty)')
    raise RuntimeError(f'Smoke check failed (exit {_r.returncode}). Try --kaggle-lite instead if this OOMs.')
print('\n✓ Smoke check PASSED (kaggle_medium fits VRAM)')

In [ ]:
# ── Full training run — kaggle_medium preset (~475K params) to fit T4 VRAM at 15,367 nodes ──
# NOTE: --run-baselines is deliberately omitted here (RF/XGBoost on ~7.9M rows is
# too slow/memory-heavy on Kaggle CPU for this node count). Persistence and
# climatology baselines are still computed automatically every epoch.
# If this OOMs, switch '--kaggle-medium' to '--kaggle-lite' below.
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir',               PROCESSED_DIR,
    '--checkpoint-dir',         CHECKPOINT_DIR,
    '--epochs',                 '100',
    '--device',                 'auto',
    '--amp',
    '--kaggle-medium',
    '--batch-size',              '1',
    '--grad-accum-steps',       '8',
    '--cosine-lr',
    '--early-stopping-patience', '20',
    '--weight-decay',           '1e-4',
    '--gnn-dropout',            '0.12',
    '--lambda-conservation',    '0.02',
    '--lambda-smoothness',      '0.02',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet — run the training cell first.')
else:
    history = json.loads(history_path.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Full India Training Loss')
    axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['epochs'], history['val_r2'], color='green', label='R2 Tmax')
    axes[1].axhline(0.80, color='red', linestyle='--', label='Target R2=0.80')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('R2'); axes[1].set_title('Validation R2')
    axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves_full_india.png', dpi=150)
    plt.show()
    print('Best val_loss:', min(history['val_loss']))
    if history['benchmark_metrics']:
        last = history['benchmark_metrics'][-1]
        print(f"Latest R2_tmax={last.get('r2_tmax'):.3f} | R2_tmin={last.get('r2_tmin'):.3f} | R2_rain={last.get('r2_rain'):.3f}")

In [ ]:
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    dst = f'/kaggle/working/vayu_best_{REGION}.pt'
    shutil.copy(best_ckpt, dst)
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: {dst} ({size_mb:.1f} MB)')
else:
    print('vayu_best.pt not found — check training cell output for errors.')